# TrialScope — ClinicalTrials.gov API Exploration

**Project:** TrialScope  
**Phase:** 1 — Data Exploration  
**Data source:** ClinicalTrials.gov API v2

## Objective

The goal of this notebook is to explore the structure and content of clinical trial records before designing the data model and extraction pipeline.

At this stage, we are **not** trying to build the final dataset. We are learning:

- how the API response is structured;
- which modules contain the information relevant to TrialScope;
- which fields may be useful for analytics;
- which fields require transformation;
- which relationships exist between trials and their associated data.

This exploration will be used as the basis for the project's **Data Dictionary** and database design.

## 1. Import dependencies

We use `requests` to communicate with the ClinicalTrials.gov API.

In [ ]:
import requests

## 2. Retrieve clinical trial data

ClinicalTrials.gov exposes study records through its API.

For the first exploration, we retrieve the studies endpoint without applying filters. The purpose is simply to inspect the response structure.

In [ ]:
url = "https://clinicaltrials.gov/api/v2/studies"

response = requests.get(url, timeout=30)

print("HTTP status:", response.status_code)

A successful HTTP status (`200`) indicates that the request was accepted by the API.

In [ ]:
data = response.json()

print("Response type:", type(data))
print("Top-level keys:", data.keys())

## 3. Explore the API response

The response contains a collection of studies and a pagination token.

This is an important observation for the future extraction pipeline: the API does **not** necessarily return the entire dataset in one request. We will need to implement pagination when we build the production extractor.

In [ ]:
print("Number of studies returned:", len(data["studies"]))
print("Next page token available:", "nextPageToken" in data)

## 4. Inspect a single study

We will use the first study returned by the API as an example record.

The objective is to understand the schema before deciding how to transform the data.

In [ ]:
study = data["studies"][0]

print("Study keys:")
print(study.keys())

Each study contains several major sections. The most important one for our initial exploration is `protocolSection`, which contains the structured information describing the clinical trial.

In [ ]:
protocol = study["protocolSection"]

print(protocol.keys())

## 5. Identification information

The identification module contains information that uniquely identifies and describes the study.

Typical fields include the NCT identifier, brief title, official title, and sponsor study ID.

The NCT identifier will become the natural external identifier used to connect records across our analytical tables.

In [ ]:
identification = protocol["identificationModule"]

print(identification)

## 6. Explore the protocol modules

Before extracting individual variables, we inspect the available modules.

This is equivalent to inspecting columns in a tabular dataset: we first need to understand what information is available before deciding what to keep.

In [ ]:
for module_name in protocol:
    print(module_name)

The modules identified in this study include:

- identification
- status
- sponsor/collaborators
- oversight
- description
- conditions
- design
- arms/interventions
- outcomes
- eligibility
- contacts/locations

These modules represent different analytical dimensions of a clinical trial.

## 7. Status module

The status module contains dates and lifecycle information.

Important fields identified:

- `overallStatus`
- `startDateStruct`
- `primaryCompletionDateStruct`
- `completionDateStruct`
- submission/posting dates

The date structures contain both a date and a type such as `ACTUAL` or `ESTIMATED`.

This distinction is important because our analytical model should not blindly treat every date as equivalent.

In [ ]:
status = protocol["statusModule"]

print("Available fields:")
print(status.keys())

print("\nCurrent study status:", status.get("overallStatus"))
print("Start date:", status.get("startDateStruct"))
print("Primary completion:", status.get("primaryCompletionDateStruct"))
print("Completion date:", status.get("completionDateStruct"))

### Initial analytical use

These fields can support questions such as:

- How has clinical trial activity changed over time?
- How long do trials take to complete?
- Do durations differ by phase or study type?
- How does study status vary across the clinical trial landscape?

## 8. Study design and enrollment

The design module describes how the trial is structured.

Important fields identified:

- `studyType`
- `phases`
- `designInfo`
- `enrollmentInfo`

Enrollment is represented by both a participant count and a type, such as actual or estimated enrollment.

In [ ]:
design = protocol["designModule"]

print("Available fields:")
print(design.keys())

print("\nStudy type:", design.get("studyType"))
print("Phases:", design.get("phases"))
print("Design information:", design.get("designInfo"))
print("Enrollment:", design.get("enrollmentInfo"))

### Initial analytical use

These variables can support analysis of:

- trial phases;
- randomized vs. non-randomized designs;
- intervention models;
- masking/blinding;
- primary purpose;
- study size.

## 9. Conditions

A trial can be associated with one or more medical conditions.

This is our first clear example of a **one-to-many relationship**:

`Trial → Conditions`

We should therefore avoid assuming that every trial has exactly one condition.

In [ ]:
conditions = protocol["conditionsModule"]

print("Available fields:", conditions.keys())
print("Conditions:", conditions.get("conditions"))

For the final model, conditions will likely be represented separately from the main trial record so that multiple conditions can be associated with the same trial.

## 10. Sponsors and collaborators

The sponsor module identifies the lead sponsor and information about the responsible party.

For analytics, sponsor name and sponsor class are especially relevant because they allow comparisons between organizations and sponsor categories.

In [ ]:
sponsor = protocol["sponsorCollaboratorsModule"]

print(sponsor)

Potential analytical questions include:

- Which organizations sponsor the most trials?
- How does trial activity differ by sponsor class?
- Are there differences in study characteristics between sponsor categories?

## 11. Interventions and arm groups

This module contains two related concepts:

- **Interventions:** drugs, devices, procedures, or other interventions being studied.
- **Arm groups:** groups of participants receiving a particular treatment or control condition.

An arm can reference one or more interventions, and an intervention can appear in multiple arm groups.

This means the relationship is more complex than a simple `trial → intervention` structure.

In [ ]:
arms_interventions = protocol["armsInterventionsModule"]

print("Available fields:", arms_interventions.keys())

print("\nInterventions:")
for intervention in arms_interventions.get("interventions", []):
    print("-", intervention.get("name"), "|", intervention.get("type"))

print("\nArm groups:")
for arm in arms_interventions.get("armGroups", []):
    print("-", arm.get("label"), "|", arm.get("type"))

This relationship will probably require separate entities/tables for trials, interventions, arm groups, and their associations.

The full arm descriptions may also become useful later for text analysis, but they are not necessarily required in the first analytical model.

## 12. Eligibility

Eligibility information describes the population that can participate in the trial.

Structured fields identified include:

- minimum age;
- maximum age;
- sex;
- healthy volunteers;
- standardized age groups.

There is also a large free-text field containing inclusion and exclusion criteria.

In [ ]:
eligibility = protocol["eligibilityModule"]

print("Available fields:", eligibility.keys())

print("Minimum age:", eligibility.get("minimumAge"))
print("Maximum age:", eligibility.get("maximumAge"))
print("Sex:", eligibility.get("sex"))
print("Healthy volunteers:", eligibility.get("healthyVolunteers"))
print("Standardized ages:", eligibility.get("stdAges"))

### Important transformation

Age is currently represented as text such as `"18 Years"`.

For analytical purposes, we will probably transform this into numeric variables such as:

- `minimum_age`
- `maximum_age`

The original eligibility criteria text should be retained separately if we later use NLP.

## 13. Locations

A trial may have multiple research locations.

Location records can contain:

- facility;
- city;
- state;
- country;
- latitude;
- longitude.

This gives TrialScope a geographic dimension and opens the possibility of spatial analysis and mapping.

In [ ]:
locations = protocol["contactsLocationsModule"]

print("Available fields:", locations.keys())

print("\nLocations:")
for location in locations.get("locations", []):
    geo = location.get("geoPoint", {})
    print(
        "-",
        location.get("facility"),
        "|",
        location.get("city"),
        "|",
        location.get("country"),
        "|",
        geo.get("lat"),
        geo.get("lon")
    )

## 14. Description

The description module contains free-text information about the study.

The two main fields identified are:

- `briefSummary`
- `detailedDescription`

These fields are not ideal for basic relational analytics, but they may become useful for NLP and similarity analysis in a later phase of TrialScope.

In [ ]:
description = protocol["descriptionModule"]

print("Available fields:", description.keys())

print("\nBrief summary:")
print(description.get("briefSummary"))

## 15. Outcomes

Outcomes describe what the trial intends to measure.

The API separates:

- primary outcomes;
- secondary outcomes.

Each outcome contains at least:

- `measure`;
- `description`;
- `timeFrame`.

This creates another one-to-many relationship:

`Trial → Outcomes`

In [ ]:
outcomes = protocol["outcomesModule"]

print("Available fields:", outcomes.keys())

print("\nPrimary outcomes:")
for outcome in outcomes.get("primaryOutcomes", []):
    print("-", outcome.get("measure"), "|", outcome.get("timeFrame"))

print("\nSecondary outcomes:")
for outcome in outcomes.get("secondaryOutcomes", []):
    print("-", outcome.get("measure"), "|", outcome.get("timeFrame"))

### Initial analytical use

Outcomes could allow us to investigate:

- which endpoints are most frequently used;
- how primary and secondary outcomes differ;
- which outcomes are common within specific conditions;
- how outcome timeframes vary between studies.

Outcome descriptions may also become an NLP dataset later.

# 16. Initial data model

Based on the exploration so far, the following entities have emerged:

- **Trial**
- **Condition**
- **Sponsor**
- **Intervention**
- **Arm Group**
- **Location**
- **Eligibility**
- **Outcome**

Several of these have one-to-many or many-to-many relationships.

A preliminary conceptual structure is:

```text
                         TRIAL
                           |
        +------------------+------------------+
        |        |         |        |        |
        v        v         v        v        v
   CONDITION  PHASE    SPONSOR  LOCATION  ELIGIBILITY
                           |
                           v
                     INTERVENTION
                           |
                           v
                       ARM GROUP

                         TRIAL
                           |
                           v
                        OUTCOME
```

This is a conceptual model only. The final relational/analytical schema will be designed after completing the Data Dictionary.

# 17. Initial analytical questions

The exploration suggests several questions that TrialScope can investigate:

### Clinical landscape

- How does clinical trial activity evolve over time?
- Which conditions have the highest number of trials?
- How are trials distributed across phases?

### Study design

- Which study designs are most common?
- How does enrollment vary by phase?
- How long do trials take to complete?

### Sponsors

- Which organizations sponsor the most studies?
- How does activity differ by sponsor class?

### Geography

- Which countries and cities host the most trial locations?
- How is clinical research distributed geographically?

### Outcomes

- Which endpoints are most frequently used?
- How do outcome types and timeframes vary by condition or phase?

### Data Science

- Can trial duration be predicted from study characteristics?
- Can clinical trials or their textual descriptions be grouped by similarity?

# 18. Key observations from the exploration

1. The API response is hierarchical JSON rather than a flat analytical table.
2. A single trial can have multiple conditions, interventions, arm groups, locations, and outcomes.
3. Several fields require transformation before they are suitable for analysis.
4. Some fields are structured and immediately useful for analytics, while others are free text and better suited to NLP.
5. API pagination will need to be handled when building the full extraction pipeline.
6. The final analytical model should not simply mirror the API structure; it should be designed around the questions TrialScope aims to answer.

These observations will guide the next phase: **Data Dictionary and data model design**.